# Random numbers and distributions

> Seeded sampling and probability functions.

In [ ]:
import rustymath as rm

## A persistent generator

`default_rng(seed)` creates one generator whose state advances across calls. A size tuple gives the output shape. Reuse the generator for different distributions:

In [ ]:
rng = rm.default_rng(42)
draws = rng.normal(size=(3, 2))
draws, draws.mean(axis=0), rng.sample(rm.distribution.beta(2, 3), 5)

(Array([[-1.0920367, -0.30040901], [-0.78924237, 0.52985647], [-2.2347094, 1.3224215]], dtype=float64),
 Array([-1.3719962, 0.51728964], dtype=float64),
 Array([0.16794595, 0.31556707, 0.25183041, 0.32702751, 0.60316242], dtype=float64))

`integers` draws from the half-open interval [low, high). `choice` and `permutation` sample or permute along the first axis, or over `range(n)` when passed an integer `n`. Here the two selected items are whole rows:

In [ ]:
rng.integers(0, 10, size=4), rng.choice(draws, size=2, replace=False)

(Array([5, 3, 0, 4], dtype=int64),
 Array([[-1.0920367, -0.30040901], [-2.2347094, 1.3224215]], dtype=float64))

The seed initializes the generator's state. It uses rustymath's random number generation, shared with its Rust core, and does not reproduce NumPy's random sequences. Draws can differ with thread count or scheduling, execution route, platform or build.

`multivariate_normal(mean, cov, size)` draws from a multivariate normal distribution by multiplying standard normal draws by the Cholesky factor of `cov`. It takes a real mean vector and a real positive-definite covariance matrix. Neither may have axis keys or names. It returns float64 samples whose final axis holds the components. If `cov` isn't positive definite, its Cholesky factor is NaN, and so are the samples.

In [ ]:
rng.multivariate_normal([1., 2.], [[4., 1.], [1., 3.]], size=3)

Array([[-0.90930908, 2.008366], [4.0249491, 6.4071364], [-1.7609395, 1.433404]], dtype=float64)

## Probability functions

`rm.distribution` constructors return objects with `pdf`, `cdf`, `quantile` and `sample`. `cdf(x)` is the probability of a value at most `x`. Subtract `cdf(x)` from one to get the probability of a value greater than `x`. Supply `seed` to start the sample from a chosen generator state:

In [ ]:
n = rm.distribution.normal()
n.cdf([-1.96, 0, 1.96]), 1 - n.cdf(1.96), n.quantile(0.975), n.sample(3, seed=0)

(Array([0.024997895, 0.5, 0.9750021], dtype=float64),
 0.024997895148220484,
 1.9599639845400538,
 Array([0.11623959, -0.78285485, -2.3346353], dtype=float64))

For a discrete distribution, `pdf(k)` gives the probability mass at `k`. With an expected rate of 2 arrivals per interval, `poisson(2)` gives the probabilities of 0 to 3 arrivals, the probability of at least 4, and the 95th-percentile count:

In [ ]:
arrivals = rm.distribution.poisson(2)
arrivals.pdf([0, 1, 2, 3]), 1 - arrivals.cdf(3), arrivals.quantile(0.95)

(Array([0.13533528, 0.27067057, 0.27067057, 0.18044704], dtype=float64),
 0.14287653950145296,
 5.0)

`rng.sample(distribution, size)` samples from any of these distributions using the persistent generator `rng`.

| Constructor | Parameters |
|---|---|
| `normal(mean=0, sd=1)` | Mean and standard deviation |
| `uniform(min=0, max=1)` | Lower and upper limits |
| `beta(a, b)` | Two shapes |
| `cauchy(location=0, scale=1)` | Location and scale |
| `chi_squared(df)` | Degrees of freedom |
| `exponential(rate=1)` | Rate |
| `fisher(df1, df2)` | Numerator and denominator degrees of freedom |
| `gamma(shape, scale)` | Shape and scale |
| `inverse_gamma(shape, scale)` | Shape and scale |
| `laplace(location=0, scale=1)` | Location and scale |
| `logistic(location=0, scale=1)` | Location and scale |
| `log_normal(location=0, scale=1)` | Mean and standard deviation of the logarithm |
| `student_t(df)` | Degrees of freedom |
| `weibull(shape, scale)` | Shape and scale |
| `binomial(n, p)` | Trial count and success probability |
| `poisson(rate)` | Mean count |